# Phase 6 & 7: Model Training, Imbalance Handling & Benchmarking
## Comparing Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM, and MLP

Key Objectives:
1. Benchmark 6 diverse model families
2. Implement class imbalance mitigation (Balanced class weighting & threshold tuning)
3. Evaluate using **Macro F1, Minority Serious Recall, and PR-AUC** (not accuracy alone)
4. Plot side-by-side ROC curves, Precision-Recall curves, and confusion matrices


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.config import REPORTS_DIR, FIGURES_DIR
from src.models import train_and_evaluate_all_models
from src.data_loader import load_dataset
from src.preprocessor import prepare_raw_data, get_feature_lists, build_preprocessor_pipeline, stratified_split

# Prepare data
df = prepare_raw_data(load_dataset(sample_size=30000))
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(df)
num_cols, cat_cols, road_cols = get_feature_lists(X_train)
preprocessor = build_preprocessor_pipeline(num_cols, cat_cols, road_cols, scale_numeric=True)

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)


### 1. Train and Benchmark All Models

In [ ]:
results_df, fitted_models, best_thresholds = train_and_evaluate_all_models(
    X_train_proc, y_train.values, X_val_proc, y_val.values, imbalance_strategy="class_weight"
)
results_df


### 2. View Performance Comparison Table

In [ ]:
comp_csv = REPORTS_DIR / "model_comparison.csv"
if comp_csv.exists():
    display_df = pd.read_csv(comp_csv)
    display(display_df.style.highlight_max(subset=['Macro F1', 'Recall (Serious)', 'ROC-AUC', 'PR-AUC'], color='#bfdbfe'))
